# 01.01 · Adquisición incremental de subtítulos

Reutiliza transcripciones canónicas y cachés por `video_id`; solo consulta YouTube para candidatos nuevos y nunca descarga audio o video.

**Contrato v2.1:** `SEGURO` + cuatro daños entrenados, incluida `ATAQUE_POR_GENERO_IDENTIDAD`. `SEGURO` es excluyente; los daños son multietiqueta. Los casos indeterminados se difieren y no entran al entrenamiento.

## Reproducibilidad

El cuaderno solo orquesta funciones versionadas de `src/moderacion_peru`. En local no instala paquetes. En Colab, únicamente la celda de bootstrap instala versiones fijadas desde el bundle SHA-256 de Drive. No usa rutas personales. Revise el README de esta etapa.

In [ ]:
from pathlib import Path
import sys

def find_root(start=Path.cwd()):
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'pyproject.toml').is_file():
            return candidate
    raise FileNotFoundError('No se encontró pyproject.toml')

ROOT = find_root()
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
print('Proyecto:', ROOT)


## Preflight

In [ ]:
from moderacion_peru.artifacts import artifact_status
artifact_status(ROOT)

## Reutilización de snapshots existentes

In [ ]:
from moderacion_peru.acquisition import (bootstrap_canonical_from_existing, discover_existing_transcript_sources, fetch_youtube_subtitles, ingest_incremental, load_candidates)
CANONICAL = ROOT/'datos/raw/transcripts_raw.jsonl'
CACHE = ROOT/'datos/raw/transcripts_cache'
sources = discover_existing_transcript_sources(ROOT, canonical_path=CANONICAL)
reuse_stats = bootstrap_canonical_from_existing(sources, CANONICAL)
print(reuse_stats)

## Candidatos y caché

In [ ]:
CANDIDATE_FILES = [ROOT/'datos/raw/video_candidates.jsonl', ROOT/'datos/raw/videos_candidatos.csv']
candidates_by_id = {}
for source in CANDIDATE_FILES:
    for row in load_candidates(source):
        candidates_by_id.setdefault(str(row['video_id']), row)
candidates = list(candidates_by_id.values())
print('Candidatos:', len(candidates), '· los ya existentes se omitirán')

## Ejecución controlada

In [ ]:
FETCH_NEW = False  # Cambie a True solo para consultar candidatos no vistos
if candidates:
    stats = ingest_incremental(candidates, CANONICAL, CACHE, fetcher=fetch_youtube_subtitles if FETCH_NEW else None)
    print(stats)
else:
    print('Agregue candidatos con video_id y url; el corpus existente no se vuelve a descargar.')